Imports:

In [1]:
import json
import os
from bs4 import BeautifulSoup
from markdown_it.rules_block import paragraph
from requests import RequestException
import re
import wikipediaapi
import requests
from pathlib import Path

In [2]:
USER_AGENT = os.getenv('Learning project: TourGuideAI/1.0', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
                          ' Chrome/91.0.4472.124 Safari/537.36')

JSON-Dateien aus dem Verzeichnis 'data' laden und in einer Liste speichern:

In [3]:
def load_data(path='../data/ausflugziele'):
    data = []
    for filename in os.listdir(path):
        # Nur JSON-Dateien verarbeiten
        if filename.endswith('.json'):
            # Öffnen und Laden der JSON-Datei im Lesemodus
            with open(os.path.join(path, filename), 'r', encoding='utf-8') as f:
                # Inhalt der JSON-Datei laden
                content = json.load(f)
                if isinstance(content, list):
                    data.extend(content)
                else:
                    data.append(content)
    return data

data = load_data()
print(f'{len(data)} documents loaded.')

17 documents loaded.


In [4]:
URL_PATH_WEB = Path('../data/web/*.json')
def load_urls(path: Path):
    urls = []
    for file in path.glob("*.json"):
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

            for entry in data:
                urls.append(entry["url"])

    return urls
urls_web = load_urls(URL_PATH_WEB)
urls_web


[]

In [5]:
URL_PATH_WIKI = Path('../data/wiki')
def load_urls(path: Path):
    urls = []
    for file in path.glob("*.json"):
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

            for entry in data:
                urls.append(entry["url"])
    return urls
urls_wiki = load_urls(URL_PATH_WIKI)
urls_wiki

['https://de.wikipedia.org/wiki/Spreewald',
 'https://de.wikipedia.org/wiki/L%C3%BCbbenau/Spreewald',
 'https://de.wikipedia.org/wiki/Brandenburg_an_der_Havel',
 'https://de.wikipedia.org/wiki/Potsdam',
 'https://de.wikipedia.org/wiki/Potsdamer_Stadtschloss',
 'https://de.wikipedia.org/wiki/Alter_Markt_(Potsdam)']

In [6]:
def get_soup(url):
    headers = {"Accept": "text/html,application/json",
               'User-Agent': USER_AGENT}
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status() #wenn !200, Fehler werfen
    return BeautifulSoup(response.text, 'html.parser')

In [7]:
def extract_opening_hours(soup):
    results = []
    for text in soup.find_all(string=True):
        if text and "uhr" in text.lower():
            cleaned = text.strip()
            if 5 < len(cleaned) < 200:
                results.append(cleaned)
    return list(set(results))

In [8]:
def process_urls(urls, date_string):
    results = {}

    for url in urls:
        try:
            soup = get_soup(url)
            results[url] = {
                "Öffnungszeiten": extract_opening_hours(soup),

            }
        except Exception as e:
            print(f"Fehler bei {url}: {e}")

    return results


In [9]:
results = process_urls(urls_web, "12.04.2026")

print(json.dumps(results, indent=4, ensure_ascii=False))


{}


In [10]:
wiki = wikipediaapi.Wikipedia(language='de',
                             extract_format=wikipediaapi.ExtractFormat.WIKI,
                             user_agent=USER_AGENT)

def load_wikipedia_text(title: str) -> dict:
    page = wiki.page(title)
    if not page.exists():
        return {'error': 'Page not found'}
    text = page.text
    return {
        'text': text,
        'title': page.title,
        'source': page.fullurl,
        'license': 'CC BY-SA 4.0'
    }

#Titel aus URL extrahieren
def extract_title_from_url(url: str) -> str:
    match = re.search(r'/wiki/([^#]+)', url)
    if match:
        return match.group(1)
    return None

def get_summary_from_wiki(text:str, n=2) -> str:
    paragraphs = text.split('\n\n')
    summary = ' '.join(paragraphs[:n])
    return summary

wiki_data = {}
for url in urls_wiki:
    title = extract_title_from_url(url)
    wiki_data[title] = load_wikipedia_text(title)


# Ergebnis als JSON ausgeben
import json
#print(json.dumps(wiki_data, indent=2, ensure_ascii=False))
print(f"\n {len(wiki_data)} Wikipedia-Artikel geladen.")
for title, data in wiki_data.items():
    if 'text' in data:
        summary = get_summary_from_wiki(data['text'])
        print(f"\n {data['title']}\n{summary}\n")


 6 Wikipedia-Artikel geladen.

 Spreewald
Der Spreewald (niedersorbisch Błota, „die Sümpfe“) ist ein ausgedehntes Niederungsgebiet und eine historische Kulturlandschaft im Südosten Brandenburgs. Hauptmerkmal ist die natürliche Flusslaufverzweigung der Spree, die durch angelegte Kanäle deutlich erweitert wurde. Als Auen- und Moorlandschaft besitzt sie für den Naturschutz überregionale Bedeutung und ist als Biosphärenreservat geschützt (siehe Biosphärenreservat Spreewald). Der Spreewald als Kulturlandschaft wurde entscheidend durch die Sorben geprägt. Das Gebiet ist eines der bekanntesten und beliebtesten Reiseziele im Land Brandenburg. Insgesamt 222,8 Kilometer im Oberspreewald und 45,4 Kilometer im Unterspreewald sind als Landeswasserstraße klassifiziert. Geografie
Gliederung
Der Spreewald befindet sich in den Landkreisen Spree-Neiße, Dahme-Spreewald und Oberspreewald-Lausitz. Er wird in den südlichen und größeren Oberspreewald und den nördlichen, kleineren Unterspreewald geteilt. Zwi

In [11]:
# urls = [
#     'https://www.museum-barberini.de/de/',
#     'https://shop.museum-barberini.de/#/tickets/time?group=timeSlot&lang=de',
#     'https://shop.museum-barberini.de/#/tickets/combi'
#     'https://shop.museum-barberini.de/#/tickets/time?group=timeSlot',
#     'https://www.museum-barberini.de/de/kalender/',
#     'https://www.industriemuseum-brandenburg.de/home'
# ]

In [12]:
# HTML von einer URL laden und in BeautifulSoup-Objekt umwandeln



def get_soup(url):
    headers = {"Accept": "text/html,application/json",
               'User-Agent': USER_AGENT}
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status() #wenn !200, Fehler werfen
    return BeautifulSoup(response.text, 'html.parser')

def extract_opening_hours(soup):
    results = []
    for text in soup.find_all(string=True):
        if text and "uhr" in text.lower():
            cleaned = text.strip() # Whitespace entfernen
            if 5 < len(cleaned) < 200: # Nur relevante Texte behalten
                results.append(cleaned)
    return list(set(results))


def extract_ticket_prices(soup):
    results = []
    price_pattern = re.compile(r"\d+[.,]?\d*\s?€")

    for text in soup.find_all(string=True):
        if text:
            matches = price_pattern.findall(text)
            results.extend(matches)

    return list(set(results))


def extract_events_by_date(soup, date_string):
    events = []
    for element in soup.find_all(["h2", "h3", "p", "li"]):
        text = element.get_text(strip=True)
        if date_string in text:
            events.append(text)
    return events


def process_urls(urls, date_string):
    results = {}

    for url in urls:
        try:
            soup = get_soup(url)
            data = {
                "Öffnungszeiten": extract_opening_hours(soup),
                "Tickets": extract_ticket_prices(soup),
                "Events": extract_events_by_date(soup, date_string)
            }
            results[url] = data

        except Exception as e:
            print(f"Fehler bei {url}: {e}")

    return results


results = process_urls(urls_web, "12.04.2026")

print(json.dumps(results, indent=4, ensure_ascii=False))

{}


In [13]:
import requests
import json
import os

USER_AGENT = os.getenv("USER_AGENT", "Mozilla/5.0 (Windows NT 10.0; Win64; x64)")

def get_ticket_data(date):
    url = (
        "https://barberini.gomus.de/api/v4/tickets"
        "?by_bookable=true"
        "&by_free_timing=false"
        "&by_ticket_type=time_slot"
        "&locale=de"
        "&per_page=1000"
        f"&valid_at={date}"
    )

    headers = {
        "Accept": "application/json",
        "x-shop-url": "shop.museum-barberini.de",
        "User-Agent": USER_AGENT
    }

    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    return response.json()

def extract_prices(data):
    results = []
    for ticket in data.get("tickets", []):

        results.append({
            "name": ticket.get("title"),
            "price in EUR": ticket.get("price_cents", 0) / 100,
            "ticket_type": ticket.get("ticket_type"),
            "free_timing": ticket.get("free_timing"),
            "bookable": ticket.get("bookable")
        })
    return results

# Beispiel: 5. März 2026
date = "2026-03-05"
data = get_ticket_data(date)
prices = extract_prices(data)

print(json.dumps(prices, indent=2, ensure_ascii=False))

[
  {
    "name": "Hausticket regulär",
    "price in EUR": 16.0,
    "ticket_type": "time_slot",
    "free_timing": false,
    "bookable": true
  },
  {
    "name": "Hausticket ermäßigt",
    "price in EUR": 10.0,
    "ticket_type": "time_slot",
    "free_timing": false,
    "bookable": true
  },
  {
    "name": "Hausticket frei (0 € Ticket)",
    "price in EUR": 0.0,
    "ticket_type": "time_slot",
    "free_timing": false,
    "bookable": true
  },
  {
    "name": "Hausticket Barberini am Abend",
    "price in EUR": 10.0,
    "ticket_type": "time_slot",
    "free_timing": false,
    "bookable": true
  },
  {
    "name": "Hausticket frei Barberini am Abend",
    "price in EUR": 0.0,
    "ticket_type": "time_slot",
    "free_timing": false,
    "bookable": true
  },
  {
    "name": "Hausticket regulär ",
    "price in EUR": 18.0,
    "ticket_type": "time_slot",
    "free_timing": false,
    "bookable": true
  },
  {
    "name": "Hausticket ermäßigt",
    "price in EUR": 10.0,
    "tic

URLs lesen und Texte aus den Dokumenten extrahieren:

In [14]:
# HTML ohne Webloader laden

def load_html(url: str) -> BeautifulSoup:
    headers = {'User-Agent': os.getenv('USER_AGENT')}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except RequestException as e:
        print(f"Fehler beim Laden von {url}: {e}")
        return None
    return BeautifulSoup(response.text, 'html.parser')

In [15]:
# Regex Patterns
ADDRESS_REGEX = re.compile(
    r"\b([A-ZÄÖÜ][\wäöüß\-]+(?:straße|strasse|weg|allee|platz|gasse|ring|damm|ufer|hof|steig|chaussee|promenade|markt|pfad))"
    r"\s+(\d+[a-zA-Z]?)"
    r"\s+(\d{5})"
    r"\s+([A-ZÄÖÜ][a-zäöüß]+(?:\s+[a-zäöüß]+)*(?:\s+[A-ZÄÖÜ][a-zäöüß]+)*)"
    r"(?=\s*(?:$|<|\.|,|;|:|\n|\bTelefon\b|\bTel\b|\bFax\b|\bE-Mail\b|\bEmail\b|\bÖffnungszeiten\b))",
    flags=re.IGNORECASE
)
ADDRESS_REGEX = re.compile(
    r"([A-ZÄÖÜ][\wäöüß\-]+\s*(?:straße|strasse|weg|allee|platz|gasse|ring|damm|ufer|hof|steig|chaussee|promenade|markt|pfad)?)\s+(\d+[a-zA-Z]?)?,?\s*(\d{5})?\s*([A-ZÄÖÜ][a-zäöüß]+(?:\s+[a-zäöüß]+)*)?",
    flags=re.IGNORECASE
)

STUNDEN_REGEX = re.compile(r"\b(?:Mo|Di|Mi|Do|Fr|Sa|So)\.?\s*(?:\d{1,2}:\d{2}\s*-\s*\d{1,2}:\d{2}|geschlossen)\b",  flags=re.IGNORECASE)
PREIS_REGEX = re.compile(r"\b(?:Eintritt|Preis|Kosten|Tarif|Gebühr).{0,20}?\b\d{1,3}(?:,\d{2})?\s*€\b",  flags=re.IGNORECASE)

In [17]:
# Facts extrahieren

def extract_facts(soup: BeautifulSoup) -> dict:
    if soup is None:
        return {}

    text = ' '.join(soup.get_text(' ').split())
    facts = {}

    adresse = ADDRESS_REGEX.search(text)
    stunden = STUNDEN_REGEX.search(text)
    preis = PREIS_REGEX.search(text)

    if adresse:
        facts['Adresse'] = adresse.group(0)
    if stunden:
        facts['Eintritt'] = stunden.group(0)
    if preis:
        facts['Preis'] = preis.group(0)

    return facts


In [18]:
#Texte bereinigen
def clean_text(text):
     # Kleinbuchstaben, Entfernen von Sonderzeichen
    text = text.lower().replace(",", " ").replace("!", " ").replace("?", " ").replace("-", " ").strip()

    text = text.split()

    return text

for title, data in wiki_data.items():
        if 'text' in data:
            print(clean_text(data['text']))
        else:                     # Fehler abfangen, sonst Error
            print(f"Fehler bei {title}: {data.get('error', 'Unbekannter Fehler')}")

['der', 'spreewald', '(niedersorbisch', 'błota', '„die', 'sümpfe“)', 'ist', 'ein', 'ausgedehntes', 'niederungsgebiet', 'und', 'eine', 'historische', 'kulturlandschaft', 'im', 'südosten', 'brandenburgs.', 'hauptmerkmal', 'ist', 'die', 'natürliche', 'flusslaufverzweigung', 'der', 'spree', 'die', 'durch', 'angelegte', 'kanäle', 'deutlich', 'erweitert', 'wurde.', 'als', 'auen', 'und', 'moorlandschaft', 'besitzt', 'sie', 'für', 'den', 'naturschutz', 'überregionale', 'bedeutung', 'und', 'ist', 'als', 'biosphärenreservat', 'geschützt', '(siehe', 'biosphärenreservat', 'spreewald).', 'der', 'spreewald', 'als', 'kulturlandschaft', 'wurde', 'entscheidend', 'durch', 'die', 'sorben', 'geprägt.', 'das', 'gebiet', 'ist', 'eines', 'der', 'bekanntesten', 'und', 'beliebtesten', 'reiseziele', 'im', 'land', 'brandenburg.', 'insgesamt', '222', '8', 'kilometer', 'im', 'oberspreewald', 'und', '45', '4', 'kilometer', 'im', 'unterspreewald', 'sind', 'als', 'landeswasserstraße', 'klassifiziert.', 'geografie', '

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
